In [19]:
import subprocess
from pathlib import Path

metadata_mashup_path = Path("metadatamashup.json")
if not metadata_mashup_path.exists():
    raise FileNotFoundError(f"{metadata_mashup_path} not found")

metadata_mashup = metadata_mashup_path.read_text()



In [20]:
#call ollama
import ollama

def analyse_metadata_mashup_query(user_query: str, metadata_mashup: str, model: str = "llama2"):
    prompt = f"""You are a API metadata matching assistant.
Given the metadata mashup below and the user query, return all the matching JSON entries with a score.

Metadata mashup:
{metadata_mashup}

User query:
{user_query}

Generate a relevance score for each entry and return only the JSON objects that match the user query. The score should be a number between 0 and 1, where 1 indicates a perfect match and 0 indicates no match.

Rules:
- Use only the metadata entries present in the mashup.
- Generate a relevance score for each entry. Tags (35%), Description (55%), category (15%) should be used to calculate the score.
- Return the top 3 JSON objects with their corresponding relevance scores.
- Exclude any entries that do not match the user query or have a score below 0.6.
- if no entries match the user query, return an empty list.

Instructions:
- Select all metadata entries that semantically match the user query.
- Return only the JSON objects for those entries.
- Show the relevance score for each entry.
- Do not add any explanation, reasoning, or extra text.
- Do not invent new entries.
"""
  
    return prompt


In [21]:
user_query = "show me last 10 cash deposits in an Branch"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
print(f"Prompt: {prompt}")

Prompt: You are a API metadata matching assistant.
Given the metadata mashup below and the user query, return all the matching JSON entries with a score.

Metadata mashup:
[
  {
    "api_id": "api_001",
    "title": "Send User Query",
    "description": "Dispatches an instantaneous query message or code to middleware.",
    "category": "User Query",
    "tags": ["Always", "Transactions", "Deposits", "Expenses", "Checks", "Cash"]
  },
  {
    "api_id": "api_002",
    "title": "Show User Deposits all sources",
    "description": "Show user deposits from all sources, including ATM, branch, mobile, and check deposits.",
    "category": "Deposits",
    "tags": ["Transactions", "Cash Deposits", "Check Deposits", "ATM", "Branch", "Mobile", "User", "Dashboard"]
  },
  {
    "api_id": "api_003",
    "title": "Show User Expenses only",
    "description": "Show user expense information for expense management and reporting.",
    "category": "Expenses",
    "tags": ["Transactions", "Expenses", "Us

In [22]:
# Call the local Ollama instance
def call_ollama(prompt: str):
    response = ollama.chat(
        model='llama3.2',
        messages=[
            { 'role': 'system', 'content': 'You are a helpful json and natural language assistant.' },
            { 'role': 'user', 'content': prompt }
        ]
    )

    # Print the response text
    #print(response['message']['content'])
    return response['message']['content']


In [23]:
user_query = "show me last 10 cash deposits in an Branch"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

```json
[
  {
    "api_id": "api_006",
    "title": "Show User Cash Deposits Branch only",
    "description": "Show branch cash deposit information for users.",
    "category": "Branch Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "Branch", "User", "Dashboard"]
  },
  {
    "api_id": "api_004",
    "title": "Show User ATM Cash Deposits only",
    "description": "Show ATM Cash deposit information for users.",
    "category": "",
    "tags": ["Transactions", "Cash Deposits", "ATM", "User", "Dashboard"]
  },
  {
    "api_id": "api_007",
    "title": "Show User Cash Withdrawals at Branch only",
    "description": "Show branch cash withdrawals information for users.",
    "category": "",
    "tags": ["Transactions", "Cash Withdrawals", "Branch", "User", "Dashboard"]
  }
]

[
  {
    "api_id": "api_006",
    "title": "Show User Cash Deposits Branch only",
    "description": "Show branch cash deposit information for users.",
    "category": "Branch Cash Deposits",
    "tags": 

In [24]:
user_query = "show me last 10 cash deposits at my bank "
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

{
  "api_004": {
    "score": 0.85,
    "title": "Show User ATM Cash Deposits only",
    "description": "Show ATM Cash deposit information for users.",
    "category": "ATM Cash Deposits"
  },
  "api_006": {
    "score": 0.75,
    "title": "Show User Cash Deposits Branch only",
    "description": "Show branch cash deposit information for users.",
    "category": "Branch Cash Deposits"
  }
}


In [25]:
user_query = "show me last 10 cash deposits at an automated teller machine"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

```json
[
  {
    "api_id": "api_004",
    "title": "Show User ATM Cash Deposits only",
    "description": "Show ATM Cash deposit information for users.",
    "category": "ATM Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "ATM", "User", "Dashboard"],
    "score": 0.93
  },
  {
    "api_id": "api_006",
    "title": "Show User Cash Deposits Branch only",
    "description": "Show branch cash deposit information for users.",
    "category": "Branch Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "Branch", "User", "Dashboard"],
    "score": 0.92
  },
  {
    "api_id": "api_005",
    "title": "Show User ATM Cash Withdrawals only",
    "description": null,
    "category": "ATM Cash Withdrawals",
    "tags": ["Transactions", "Cash Withdrawals", "ATM", "User", "Dashboard"],
    "score": 0.7
  }
]
```

Note: The scores are calculated based on the given rules, using a weighted average of the tags and description. A perfect match is assumed to have all the required ke

In [26]:
user_query = "show me last 10 expenses"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

```json
[
  {
    "api_id": "api_003",
    "title": "Show User Expenses only",
    "description": "Show user expense information for expense management and reporting.",
    "category": "Expenses",
    "tags": ["Transactions", "Expenses", "User", "Debit Card", "Credit Card", "Dashboard"]
  },
  {
    "api_id": "api_006",
    "title": "Show User Cash Deposits Branch only",
    "description": null,
    "category": "Branch Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "Branch", "User", "Dashboard"]
  }
]

```

The returned JSON objects match the user query and have a relevance score above 0.6:

- api_003: 1
- api_006: 0.928


In [27]:
user_query = "show me last 10 check deposits in an ATM"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

```json
[
  {
    "api_id": "api_004",
    "title": "Show User ATM Cash Deposits only",
    "description": "Show ATM Cash deposit information for users.",
    "category": "ATM Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "ATM", "User", "Dashboard"],
    "score": 0.8
  },
  {
    "api_id": "api_006",
    "title": "Show User Cash Deposits Branch only",
    "description": "Show branch cash deposit information for users.",
    "category": "",
    "tags": ["Transactions", "Cash Deposits", "Branch", "User", "Dashboard"],
    "score": 0.2
  },
  {
    "api_id": "api_007",
    "title": "Show User Cash Withdrawals at Branch only",
    "description": "Show branch cash withdrawals information for users.",
    "category": "",
    "tags": ["Transactions", "Cash Withdrawals", "Branch", "User", "Dashboard"],
    "score": 0.2
  }
]
```


In [29]:
user_query = "where is my money going"
prompt = analyse_metadata_mashup_query(user_query, metadata_mashup)
response = call_ollama(prompt)
print(response)

```json
[
  {
    "api_id": "api_005",
    "title": "Show User ATM Cash Withdrawals only",
    "description": "Show ATM cash withdrawal requests from users.",
    "category": "ATM Cash Withdrawals",
    "tags": ["Transactions", "Cash Withdrawals", "ATM", "User", "Dashboard"]
  },
  {
    "api_id": "api_004",
    "title": "Show User ATM Cash Deposits only",
    "description": "Show ATM Cash deposit information for users.",
    "category": "ATM Cash Deposits",
    "tags": ["Transactions", "Cash Deposits", "ATM", "User", "Dashboard"]
  },
  {
    "api_id": "api_007",
    "title": "Show User Cash Withdrawals at Branch only",
    "description": "Show branch cash withdrawals information for users.",
    "category": "Branch Cash Withdrawals",
    "tags": ["Transactions", "Cash Withdrawals", "Branch", "User", "Dashboard"]
  }
]
```

Relevance scores:
- api_005: 0.8
- api_004: 0.7
- api_007: 0.6
